# Executive Conflict Resolution — TRL GRPO Training

Trains `Qwen/Qwen2.5-1.5B-Instruct` against the Executive Conflict Resolution OpenEnv using GRPO.

**Before running:** set your `HF_TOKEN` and `ENV_URL` below.

In [ ]:
# Install dependencies
!pip install -q trl>=0.12.0 transformers accelerate peft requests openai

In [ ]:
import os
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'
os.environ['ENV_URL'] = 'https://notAathi-OpenEvnHack.hf.space'  # your live HF Space URL

ENV_URL = os.environ['ENV_URL']
HF_TOKEN = os.environ['HF_TOKEN']

In [ ]:
import requests, json, uuid

def env_reset(task_id='hard', session_id=None):
    sid = session_id or str(uuid.uuid4())
    r = requests.post(f'{ENV_URL}/reset', json={'session_id': sid, 'task_id': task_id})
    return r.json(), sid

def env_step(action: dict, session_id: str):
    r = requests.post(f'{ENV_URL}/step', json={'session_id': session_id, 'action': action})
    return r.json()

def env_score(session_id: str):
    r = requests.get(f'{ENV_URL}/score/{session_id}')
    return r.json().get('final_score', 0.0)

# Smoke test
obs, sid = env_reset('hard')
print(f'Connected. Items: {len(obs["items"])}, first: {obs["items"][0]["title"]}')

In [ ]:
SYSTEM_PROMPT = """You are an expert executive assistant resolving workplace conflicts.

For each conflict item respond with a JSON object only:
- item_id: the id of the conflict item
- conflict_type: one of scheduling, deadline, delegation, social
- resolution: one of reschedule, decline, delegate, accept, escalate
- message: the actual message to send to resolve this conflict

Example: {"item_id": "c1", "conflict_type": "scheduling", "resolution": "reschedule", "message": "Hi, I have a conflict at that time. Can we move to 4 PM?"}"""

def make_prompt(item: dict, context: str, instructions: str) -> str:
    return (
        f"{context}\n\n{instructions}\n\n"
        f"Resolve this conflict:\n"
        f"ID: {item['id']}\n"
        f"Title: {item['title']}\n"
        f"Description: {item['description']}\n"
        f"Participants: {', '.join(item['participants'])}\n"
        f"Urgency: {item['urgency']}"
    )

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=HF_TOKEN,
)
print('Model loaded:', MODEL_ID)

In [ ]:
# Collect baseline rewards before training
import json

def parse_action(raw: str, item_id: str) -> dict:
    raw = raw.strip()
    start, end = raw.find('{'), raw.rfind('}') + 1
    if start == -1 or end == 0:
        return {'item_id': item_id, 'conflict_type': 'scheduling', 'resolution': 'reschedule', 'message': ''}
    try:
        d = json.loads(raw[start:end])
        return {
            'item_id': d.get('item_id', item_id),
            'conflict_type': d.get('conflict_type', 'scheduling'),
            'resolution': d.get('resolution', 'reschedule'),
            'message': d.get('message', ''),
        }
    except Exception:
        return {'item_id': item_id, 'conflict_type': 'scheduling', 'resolution': 'reschedule', 'message': ''}

def run_episode(task_id='hard') -> float:
    obs, sid = env_reset(task_id)
    done = False
    while not done and obs.get('items'):
        item = obs['items'][0]
        prompt = make_prompt(item, obs['context'], obs['instructions'])
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True)
        raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        action = parse_action(raw, item['id'])
        result = env_step(action, sid)
        obs = result['observation']
        done = result['done']
    return env_score(sid)

# Baseline over 5 episodes
baseline_scores = [run_episode('hard') for _ in range(5)]
print(f'Baseline hard scores: {baseline_scores}')
print(f'Baseline mean: {sum(baseline_scores)/len(baseline_scores):.4f}')

In [ ]:
from datasets import Dataset

def collect_grpo_samples(n=200, task_id='hard'):
    """Collect prompt samples from the environment for GRPO training."""
    samples = []
    while len(samples) < n:
        obs, sid = env_reset(task_id)
        for item in obs.get('items', []):
            prompt = make_prompt(item, obs['context'], obs['instructions'])
            messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            samples.append({
                'prompt': text,
                'item_id': item['id'],
                'session_id': sid,
                'item': json.dumps(item),
            })
    return Dataset.from_list(samples[:n])

dataset = collect_grpo_samples(200)
print(f'Dataset size: {len(dataset)}')

In [ ]:
from trl import GRPOConfig, GRPOTrainer

reward_log = []  # track rewards per step

def reward_fn(completions, prompts=None, **kwargs):
    """Reward function: call the environment step for each completion."""
    rewards = []
    for i, completion in enumerate(completions):
        try:
            item_id = kwargs.get('item_id', ['c1'] * len(completions))[i]
            # Fresh episode per reward call to get accurate per-item reward
            obs, sid = env_reset('hard')
            # Find matching item
            target_item = next((it for it in obs['items'] if it['id'] == item_id), obs['items'][0])
            action = parse_action(completion, target_item['id'])
            result = env_step(action, sid)
            reward = result['reward']['value']
        except Exception:
            reward = 0.01
        rewards.append(float(reward))
    reward_log.extend(rewards)
    return rewards

# max_new_tokens moved to generation_config in newer TRL
model.generation_config.max_new_tokens = 200

training_args = GRPOConfig(
    output_dir='./conflict-resolution-grpo',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    logging_steps=5,
    save_steps=50,
    report_to='none',
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=reward_fn,
)

print('Starting GRPO training...')
trainer.train()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot reward curve
window = 10
smoothed = np.convolve(reward_log, np.ones(window)/window, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(reward_log, alpha=0.3, color='steelblue', label='raw reward')
plt.plot(range(window-1, len(reward_log)), smoothed, color='steelblue', linewidth=2, label=f'{window}-step avg')
plt.xlabel('Training Step')
plt.ylabel('Reward')
plt.title('Executive Conflict Resolution — GRPO Training Reward Curve')
plt.legend()
plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150)
plt.show()
print('Saved reward_curve.png')

In [ ]:
# Post-training evaluation
after_scores = [run_episode('hard') for _ in range(5)]
print(f'Before training: {sum(baseline_scores)/len(baseline_scores):.4f}')
print(f'After training:  {sum(after_scores)/len(after_scores):.4f}')
print(f'Improvement:     +{(sum(after_scores)-sum(baseline_scores))/len(baseline_scores):.4f}')

In [ ]:
# Before vs After bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Before', 'After'],
            [sum(baseline_scores)/len(baseline_scores), sum(after_scores)/len(after_scores)],
            color=['#e74c3c', '#2ecc71'], width=0.4)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Mean Final Score')
axes[0].set_title('Hard Task — Before vs After Training')

axes[1].plot(reward_log, alpha=0.3, color='steelblue')
axes[1].plot(range(window-1, len(reward_log)), smoothed, color='steelblue', linewidth=2)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Reward')
axes[1].set_title('Reward Curve During Training')

plt.tight_layout()
plt.savefig('training_summary.png', dpi=150)
plt.show()
print('Saved training_summary.png')

In [ ]:
# Save trained model to HF Hub
from huggingface_hub import login
login(token=HF_TOKEN)

model.push_to_hub('notAathi/conflict-resolution-grpo', token=HF_TOKEN)
tokenizer.push_to_hub('notAathi/conflict-resolution-grpo', token=HF_TOKEN)
print('Model pushed to HF Hub: notAathi/conflict-resolution-grpo')